# Day 2 — Document Loaders

---

Yesterday our knowledge base was a hard-coded Python list. Today we'll load **real documents** — PDFs, Word files, web pages, markdown — into the same Chroma pipeline.

You'll also learn a small design pattern (`LoaderRegistry`) that keeps the code clean as you support more file types.


## 1. What a "loader" actually does

A loader is a function with one job:

> Given a file path (or URL), return the raw text.

That's it. The chunking, embedding, and storing happens later in the pipeline — same code as Section 5.

Different file formats need different parsing libraries:

| Type | Library | Notes |
|---|---|---|
| PDF | `pypdf` | Text-based PDFs only (scanned PDFs need OCR — out of scope) |
| DOCX | `python-docx` | Word documents |
| Markdown / txt | Built-in `open()` | Just read the file |
| HTML / web page | `trafilatura` | Strips ads, navigation, boilerplate |


In [ ]:
!pip install pypdf python-docx trafilatura --quiet

## 2. PDF loader (from Section 5)


In [ ]:
from pypdf import PdfReader

def load_pdf(path: str) -> str:
    reader = PdfReader(path)
    parts = []
    for page in reader.pages:
        text = (page.extract_text() or "").strip()
        if text:
            parts.append(text)
    return "\n\n".join(parts)

# text = load_pdf("some_file.pdf")
# print(text[:400])


## 3. DOCX loader


In [ ]:
from docx import Document

def load_docx(path: str) -> str:
    doc = Document(path)
    return "\n\n".join(p.text for p in doc.paragraphs if p.text.strip())

# text = load_docx("resume.docx")
# print(text[:400])


## 4. Web page loader

`trafilatura` is the go-to for clean web extraction — it strips out headers, sidebars, ads, footers, and "related links." Way cleaner than raw BeautifulSoup for article-like pages.


In [ ]:
import trafilatura

def load_url(url: str) -> str:
    html = trafilatura.fetch_url(url)
    if not html:
        return ""
    return trafilatura.extract(html) or ""

# text = load_url("https://en.wikipedia.org/wiki/FastAPI")
# print(text[:600])


## 5. Markdown / plain text loader


In [ ]:
from pathlib import Path

def load_text(path: str) -> str:
    return Path(path).read_text(encoding="utf-8")

# text = load_text("README.md")


## 6. The `LoaderRegistry` pattern

Now imagine your ingest function has to handle any file type. You could write:

```python
if path.endswith(".pdf"): load_pdf(path)
elif path.endswith(".docx"): load_docx(path)
elif path.endswith(".md"): load_text(path)
...
```

That gets ugly fast. A cleaner pattern: **register loaders in a dict, dispatch by extension.**


In [ ]:
from pathlib import Path
from typing import Callable

LOADERS: dict[str, Callable[[str], str]] = {
    ".pdf":  load_pdf,
    ".docx": load_docx,
    ".md":   load_text,
    ".txt":  load_text,
}

def load(path: str) -> str:
    ext = Path(path).suffix.lower()
    if path.startswith("http"):
        return load_url(path)
    loader = LOADERS.get(ext)
    if loader is None:
        raise ValueError(f"No loader registered for extension: {ext}")
    return loader(path)

# print(load("README.md")[:200])
# print(load("https://en.wikipedia.org/wiki/FastAPI")[:200])


**Why this pattern is nice:**

- Adding a new format (`.html`, `.epub`, `.pptx`) is one line — no `if/elif` chain to edit.
- All the messy parsing lives in small, testable functions.
- The rest of the pipeline never has to care about file types.

This is exactly how LangChain and LlamaIndex organize their loaders internally.


## 7. Cleaning up loaded text

Loaders give you raw text. Before chunking + embedding, do a quick clean:

- Collapse runs of whitespace: `"\n\n\n\n"` → `"\n\n"`
- Strip leading/trailing whitespace
- Drop obviously junk lines (page numbers, "Chapter 3" headers if you want)

Keep it simple. Over-cleaning throws away real content.


In [ ]:
import re

def clean(text: str) -> str:
    text = re.sub(r"[ \t]+", " ", text)         # collapse spaces/tabs
    text = re.sub(r"\n\s*\n\s*\n+", "\n\n", text)  # collapse blank lines
    return text.strip()

messy = "Hello    world!\n\n\n\n\nNext paragraph.  \n\n\n"
print(repr(clean(messy)))


## 8. Full ingest pipeline — load → clean → chunk → embed → store


In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.Client()
kb = client.get_or_create_collection("multi_source_kb")


def recursive_chunk(text: str, chunk_size: int = 500, overlap: int = 50):
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks, current = [], ""
    for para in paragraphs:
        if len(current) + len(para) + 1 <= chunk_size:
            current = (current + "\n\n" + para).strip()
        else:
            if current:
                chunks.append(current)
            while len(para) > chunk_size:
                chunks.append(para[:chunk_size])
                para = para[chunk_size - overlap:]
            current = para
    if current:
        chunks.append(current)
    return chunks


def ingest(path_or_url: str) -> int:
    text = clean(load(path_or_url))
    chunks = recursive_chunk(text)
    if not chunks:
        return 0
    ids = [f"{path_or_url}::{i}" for i in range(len(chunks))]
    kb.add(
        documents=chunks,
        embeddings=model.encode(chunks).tolist(),
        metadatas=[{"source": path_or_url} for _ in chunks],
        ids=ids,
    )
    return len(chunks)

# Try any real file you have:
# n = ingest("README.md")
# print(f"Added {n} chunks. KB total: {kb.count()}")


## Recap

- A loader = *file/URL → raw text*. That's the whole job.
- Use **`pypdf`** for PDFs, **`python-docx`** for Word, **`trafilatura`** for web pages, `open()` for text/markdown.
- The **`LoaderRegistry` dict pattern** dispatches by file extension — clean and extensible.
- Always do a **light text-cleaning step** before chunking. Don't overdo it.
- **Next class:** making retrieval smarter — reranking and query rewriting.
